# SCM-VLM: train on Google Colab

Before you start: **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.

Checkpoints, the feature cache and results are stored in your Google Drive (`MyDrive/scm_vlm/`),
so nothing is lost if Colab disconnects. The dataset (~13 GB) is downloaded fresh each session
(about 5-10 minutes).

> **SCM-VLM-Discrete-Graph project.** This notebook belongs to the separate copy with the
> Discrete Causal Graph layer. Cell 3 clones the *original* project from GitHub; for this project
> run the cell below **instead of cell 3** (it unzips `SCM-VLM-Discrete-Graph.zip` from your
> Google Drive), then continue with cell 4. Section 12 trains and compares the new layer.

In [ ]:
# Use THIS project instead of cloning the original (upload SCM-VLM-Discrete-Graph.zip to MyDrive first)
%cd /content
!rm -rf scm && mkdir scm && unzip -q /content/drive/MyDrive/SCM-VLM-Discrete-Graph.zip -d scm
%cd /content/scm
!ls

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("CUDA available:", torch.cuda.is_available())

## 2. Mount Google Drive (keeps checkpoints between sessions)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/scm_vlm'
import os
for sub in ['checkpoints', 'cache', 'results']:
    os.makedirs(f'{DRIVE_DIR}/{sub}', exist_ok=True)

## 3. Get the code

Push your latest code to GitHub first (from your laptop: `git add -A`, `git commit`, `git push`).
If the repository is **private**, replace the URL with
`https://<your-github-token>@github.com/MANDEEP-SAI-SURYA/Intervention-Aware-Multimodal-Reasoning-using-SCM.git`
(create a token at GitHub → Settings → Developer settings → Personal access tokens).

In [ ]:
REPO = 'https://github.com/MANDEEP-SAI-SURYA/Intervention-Aware-Multimodal-Reasoning-using-SCM.git'
%cd /content
!rm -rf scm && git clone -q $REPO scm
%cd /content/scm
!git log --oneline | head -3

## 4. Install dependencies (PyTorch is already installed on Colab)

In [ ]:
!pip install -q transformers timm networkx seaborn openai

## 5. Keep checkpoints / cache / results on Drive

In [ ]:
%cd /content/scm
!rm -rf checkpoints cache results
!ln -s $DRIVE_DIR/checkpoints checkpoints
!ln -s $DRIVE_DIR/cache cache
!ln -s $DRIVE_DIR/results results
!ls -l | grep -- '->'

## 6. Download VQA v2 + COCO train2014 (~13 GB, 5-10 min)

The COCO object annotations (`instances_train2014.json`) are also downloaded; they are needed
for the counterfactual-supervision step later.

If a download fails (404/403), get the current links from https://visualqa.org/download.html (*Training questions* and *Training annotations*, VQA v2) and https://cocodataset.org/#download (*2014 Train images* and *2014 Train/Val annotations*), and paste them into the cell below.

In [ ]:
%cd /content/scm
!mkdir -p coco/vqa coco/train2014 coco/annotations
# VQA v2 questions + answers
!wget -q --show-progress https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Train_mscoco.zip -O /content/q.zip
!wget -q --show-progress https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Train_mscoco.zip -O /content/a.zip
!unzip -q -o /content/q.zip -d coco/vqa && unzip -q -o /content/a.zip -d coco/vqa
# COCO object annotations
!wget -q --show-progress http://images.cocodataset.org/annotations/annotations_trainval2014.zip -O /content/ann.zip
!unzip -q -o /content/ann.zip -d coco
# COCO train2014 images  ->  coco/train2014/train2014/*.jpg
!wget -q --show-progress http://images.cocodataset.org/zips/train2014.zip -O /content/train2014.zip
!unzip -q -o /content/train2014.zip -d coco/train2014 && rm /content/train2014.zip
!ls coco/vqa coco/annotations | head; ls coco/train2014/train2014 | wc -l

## 7. Check that questions and images match

In [ ]:
!python validate_dataset.py

## 8. Train

Short run first (step 1 of the plan). The first epoch runs DETR on every image and fills the cache
on Drive; later epochs (and later sessions) reuse it and are much faster.

For the full run later: `!python train.py --samples 20000 --epochs 10`

In [ ]:
!python train.py --samples 3000 --epochs 5

In [ ]:
import pandas as pd
display(pd.read_csv('results/training_history.csv'))
display(pd.read_csv('results/final_metrics.csv'))

## 9. Try the trained model on any image

In [ ]:
!python main.py --image images/Man_riding_bicycle.png --question "What is the man riding?"
!python whatif_qa.py --image images/Man_riding_bicycle.png --whatif "what if the bicycle were a motorcycle, what is the man riding?"

To use your own picture: upload it with the folder icon on the left (into `/content/scm/images/`),
then change the `--image` path above.

## 10. Copy the checkpoint to your laptop

It is already in Google Drive at `MyDrive/scm_vlm/checkpoints/scm_vlm_latest.pt`.
Download it from Drive and put it in the project's `checkpoints/` folder, or run the cell below.

In [ ]:
from google.colab import files
files.download('checkpoints/scm_vlm_latest.pt')

---
# 11. Relation-aware model with counterfactual supervision

Run cells 1-7 first (GPU, Drive, clone the **latest** code, install, links, dataset download).
This section needs `coco/annotations/instances_train2014.json` (downloaded in cell 6).

What is new in this version (see README):
* DETR features now come from the query that produced each detection (the old features were
  misaligned). The feature cache is versioned, so **the first run rebuilds `cache/detr`**
  (DETR runs once over all images, roughly 15-25 min for 30000 questions on a T4).
* Nodes get their class and box; messages depend on the pair geometry.
* Counterfactual examples with known answers are generated from the COCO annotations
  (existence, counting, replacement, irrelevant) and cached in `cache/counterfactual`.

Expect roughly 1.5-3 hours per run of `--samples 30000 --epochs 10` on a T4 (each epoch has
~27k VQA questions plus ~27k counterfactual examples). Keep the tab open; with Drive mounted the
best checkpoint and the results survive a disconnect. Use fewer epochs if you are short on time.

In [ ]:
%cd /content/scm
import os
assert os.path.isfile('coco/annotations/instances_train2014.json'), 'run cell 6 (dataset download) first'
!git log --oneline | head -3
!python -m pytest -q tests

## 11a. Full model (graph + relation-aware messages + supervised counterfactual loss)

In [ ]:
!python train.py --samples 30000 --epochs 10 --tag full

## 11b. Same model without the graph (ablation)

Identical inputs (class + box), identical counterfactual training; only the graph and the
message passing are removed. Same image-level split as 11a.

In [ ]:
!python train.py --samples 30000 --epochs 10 --graph-mode none --tag nograph

## 11c. Graph knock-out (scrambled graph) and the full evaluation

`train.py` already runs the full evaluation on the best epoch at the end of each run.
`relation_analysis.py` re-runs it on any checkpoint (same split): factual accuracy by answer
type and relational vs. non-relational questions, counterfactual accuracy per type, the
knock-out with the **learned graph / messages off / scrambled graph** (accuracy, % answers
changed, McNemar p-values), and relation probes with bootstrap 95 % CIs.

In [ ]:
!python relation_analysis.py --checkpoint checkpoints/scm_vlm_full.pt --samples 30000 --tag full
!python relation_analysis.py --checkpoint checkpoints/scm_vlm_nograph.pt --samples 30000 --tag nograph

Optional: reproduce the previous loss (the KL "don't change" term) for comparison, and repeat a
run with other seeds (`--seed` changes the initialisation only; the split stays the same):

```
!python train.py --samples 30000 --epochs 10 --cf-mode invariance_old --tag oldcf
!python train.py --samples 30000 --epochs 10 --seed 1 --tag full_s1
!python train.py --samples 30000 --epochs 10 --graph-mode none --seed 1 --tag nograph_s1
```

## 11d. Compare the runs

In [ ]:
import json, pandas as pd

def summary_row(tag):
    s = json.load(open(f'results/eval_summary_{tag}.json'))
    row = {'run': tag, 'factual': s['factual_accuracy']['overall']}
    for key, value in s['factual_accuracy'].items():
        if key.startswith('answer_type=') or key in ('relational', 'non_relational'):
            row[key] = value
    cf = s.get('counterfactual_accuracy', {})
    for key in ('overall', 'existence', 'counting', 'replacement', 'irrelevant'):
        row[f'cf_{key}'] = cf.get(key)
    ko = s.get('graph_knockout', {})
    for variant in ('messages_off', 'scrambled'):
        if isinstance(ko.get(variant), dict):
            row[f'{variant}_acc'] = ko[variant]['accuracy']
            row[f'{variant}_changed_%'] = ko[variant]['answers_changed_pct']
            row[f'{variant}_mcnemar_p'] = ko[variant]['mcnemar_vs_learned']['p_value']
    probes = s.get('relation_probes', {})
    for key in ('changed_pct_relevant', 'changed_pct_irrelevant', 'answer_object_attention_ratio'):
        if key in probes:
            row[key] = f"{probes[key]['mean']:.2f} [{probes[key]['ci_low']:.2f}, {probes[key]['ci_high']:.2f}]"
    return row

tags = [t for t in ['full', 'nograph', 'oldcf'] if os.path.isfile(f'results/eval_summary_{t}.json')]
display(pd.DataFrame([summary_row(t) for t in tags]).set_index('run').T)

In [ ]:
!zip -q -r /content/scm_results.zip results
from google.colab import files
files.download('/content/scm_results.zip')
files.download('checkpoints/scm_vlm_full.pt')

---
# 12. Discrete Causal Graph layer vs. the soft graph

Same data, same split, same counterfactual training; only `--graph-mode` differs.
Each run is roughly 1.5-3 h on a T4 at `--samples 30000 --epochs 10` (the DETR cache from
section 11 is reused if it exists).

In [ ]:
!python train.py --samples 30000 --epochs 10 --graph-mode soft --tag soft

In [ ]:
!python train.py --samples 30000 --epochs 10 --graph-mode discrete --tag discrete

Optional: no-graph baseline and extra seeds (to report mean ± std):

```
!python train.py --samples 30000 --epochs 10 --graph-mode none --tag nograph
!python train.py --samples 30000 --epochs 10 --graph-mode soft --seed 1 --tag soft_s1
!python train.py --samples 30000 --epochs 10 --graph-mode discrete --seed 1 --tag discrete_s1
```

In [ ]:
import json, os, pandas as pd

def row(tag):
    s = json.load(open(f'results/eval_summary_{tag}.json'))
    r = {'run': tag, 'factual acc': s['factual_accuracy']['overall'],
         'relational acc': s['factual_accuracy'].get('relational'),
         'counterfactual acc': s.get('counterfactual_accuracy', {}).get('overall')}
    for kind in ('existence', 'counting', 'replacement', 'irrelevant'):
        r[f'cf {kind}'] = s.get('counterfactual_accuracy', {}).get(kind)
    g = s.get('graph_structure', {})
    for key in ('dag', 'acyclicity_h', 'root_fraction', 'parents_per_node', 'stability', 'no_parent_weight'):
        if key in g:
            r[key] = g[key]['mean']
    ko = s.get('graph_knockout', {})
    for variant in ('messages_off', 'scrambled'):
        if isinstance(ko.get(variant), dict):
            r[f'{variant} changed %'] = ko[variant]['answers_changed_pct']
            r[f'{variant} McNemar p'] = ko[variant]['mcnemar_vs_learned']['p_value']
    return r

tags = [t for t in ['soft', 'discrete', 'nograph', 'soft_s1', 'discrete_s1'] if os.path.isfile(f'results/eval_summary_{t}.json')]
display(pd.DataFrame([row(t) for t in tags]).set_index('run').T)

In [ ]:
!zip -q -r /content/discrete_results.zip results
from google.colab import files
files.download('/content/discrete_results.zip')
files.download('checkpoints/scm_vlm_discrete.pt')

---
# 13. Future work: adaptive loss + curriculum, neuro-symbolic knowledge

All runs use the discrete graph (section 12) and the same data / split. Each run adds components:

| tag | adds |
|---|---|
| `discrete` | baseline (section 12, reuse it) |
| `adaptive` | `--loss-weighting uncertainty --curriculum full` |
| `neurosym` | `--knowledge-prior builtin --symbolic-fusion` |
| `fw_all` | all four |

`--knowledge-prior builtin` builds the knowledge base from `instances_train2014.json`,
with the validation images excluded. It takes about 1 minute.

In [ ]:
# optional: build and inspect the knowledge base once (prints the strongest symbolic triples)
!python build_knowledge_base.py --coco-instances coco/annotations/instances_train2014.json --exclude-val --show 20

In [ ]:
!python train.py --samples 30000 --epochs 10 --graph-mode discrete --loss-weighting uncertainty --curriculum full --tag adaptive

In [ ]:
!python train.py --samples 30000 --epochs 10 --graph-mode discrete --knowledge-prior builtin --symbolic-fusion --tag neurosym

In [ ]:
!python train.py --samples 30000 --epochs 10 --graph-mode discrete --loss-weighting uncertainty --curriculum full --knowledge-prior builtin --symbolic-fusion --tag fw_all

Compare the runs. Only report a gain if it is larger than the bootstrap CI and the
seed-to-seed spread. Use `--seed 1` / `--seed 2` with tags like `fw_all_s1` for extra seeds.

In [ ]:
import json, os, pandas as pd

def row(tag):
    s = json.load(open(f'results/eval_summary_{tag}.json'))
    cf = s.get('counterfactual_accuracy', {})
    r = {'run': tag, 'factual acc': s['factual_accuracy']['overall'],
         'relational acc': s['factual_accuracy'].get('relational'),
         'counterfactual acc': cf.get('overall'),
         **{f'cf {t}': cf.get(t) for t in ['existence', 'counting', 'replacement', 'irrelevant']}}
    g = s.get('graph_structure', {})
    for k in ['dag', 'root_fraction', 'parents_per_node', 'stability']:
        if k in g: r[k] = g[k]['mean']
    return r

tags = [t for t in ['discrete', 'adaptive', 'neurosym', 'fw_all'] if os.path.isfile(f'results/eval_summary_{t}.json')]
display(pd.DataFrame([row(t) for t in tags]).round(2))
for t in tags[1:]:
    h = pd.read_csv(f'results/training_history_{t}.csv')
    print(t, '- per-epoch loss weights / curriculum:')
    display(h[['epoch', 'competence', 'curriculum_stage', 'loss_weights', 'val_factual_accuracy', 'val_cf_accuracy']])

In [ ]:
!zip -q -r /content/future_work_results.zip results
from google.colab import files
files.download('/content/future_work_results.zip')
files.download('checkpoints/scm_vlm_fw_all.pt')